In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import re

In [ ]:
QUESTIONNAIRE = "BFI"  # "BFI" or "SD3"

df = pd.read_csv("../../outputs/tables/variation_partitioning.csv")
df["questionnaire"] = df["item"].apply(lambda x: "BFI" if "bfi" in x.lower() else "SD3")
df = df[df["questionnaire"] == QUESTIONNAIRE].copy()

In [ ]:
df["artifact_db_r2"] = df["persona_wording"] + df["task_instruction"] + df["item_wording"] + df["option_symbol"]
df["explained_db_r2"] = df["baseline_persona"] + df["artifact_db_r2"]

df["asv"] = np.where(
    df["explained_db_r2"] > 0,
    df["artifact_db_r2"] / df["explained_db_r2"],
    np.nan
)

def extract_size(model_name):
    # Matches numbers (including decimals) immediately followed by 'B' or 'b'
    # E.g., "Qwen3-0.6B" -> 0.6, "Llama-3.2-1B-Instruct" -> 1.0, "gemma-4-E2B-it" -> 2.0
    match = re.search(r'(\d+(?:\.\d+)?)[Bb]', model_name)
    return float(match.group(1)) if match else 0.0

def extract_family(model_name):
    name = model_name.lower()
    if 'llama' in name: return 'Llama'
    if 'qwen' in name: return 'Qwen'
    if 'gemma' in name: return 'Gemma'
    if 'ministral' in name: return 'Ministral'
    return 'Other'

df["size"] = df["model"].apply(extract_size)
df["family"] = df["model"].apply(extract_family)

# Clean Labels for the X-axis (e.g., "0.6B\nQwen")
df["model_label"] = df.apply(lambda row: f"{row['size']:g}B\n{row['family']}", axis=1)

# 3. Sort by family, then size so the X-axis progresses logically
df = df.sort_values(by=["family", "size"])

In [ ]:
# rename models
model_name_mapping = {
    "Llama-3.2-1B-Instruct": "1B\nLlama 3.2",
    "Llama-3.2-3B-Instruct": "3B\nLlama 3.2",
    "Llama-3.1-8B-Instruct": "8B\nLlama 3.1",
    "Qwen3-0.6B": "0.6B\nQwen 3",
    "Qwen3-1.7B": "1.7B\nQwen 3",
    "Qwen3-4B": "4B\nQwen 3",
    "Qwen3-8B": "8B\nQwen 3",
    "Qwen3-14B": "14B\nQwen 3",
    "gemma-4-E2B-it": "2B\nGemma 4",
    "gemma-4-E4B-it": "4B\nGemma 4",
    "Ministral-3-3B-Instruct-2512-BF16": "3B\nMinistral 3",
    "Ministral-3-8B-Instruct-2512-BF16": "8B\nMinistral 3",
    "Ministral-3-14B-Instruct-2512-BF16": "14B\nMinistral 3",
}

df["model_label"] = df["model"].map(model_name_mapping).fillna(df["model_label"])

In [ ]:
df["model"].unique()

In [ ]:
# sort by the following order: llama, qwen, ministral, gemma
family_order = ["Llama", "Qwen", "Ministral", "Gemma"]
df["family"] = pd.Categorical(df["family"], categories=family_order, ordered=True)
df = df.sort_values(by=["family", "size"])

In [ ]:
# 1. Setup Figure with Shared X-Axis
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(6, 5), sharex=True, 
                               gridspec_kw={'height_ratios': [2, 3]})

plt.rcParams.update({'font.family': 'serif', 'font.size': 8})

# Components: baseline_persona, persona_wording, task_instruction, item_wording, option_symbol
components = ["baseline_persona", "persona_wording", "task_instruction", "item_wording", "option_symbol"]

# Mean db-R² per component
df_plot = df.groupby("model_label", sort=False)[components].mean()
df_plot["Unexplained"] = 1.0 - df_plot[components].sum(axis=1)

# Component colors + light grey for the unexplained variation (interactions between components)
colors = ["#757575", "#26A69A", "#EF5350", "#5C6BC0", "#FFA726", "#F5F5F5"]

# --- TOP: Total variation composition (stacked bar) ---
df_plot.plot(kind='bar', stacked=True, color=colors, ax=ax1, 
             width=0.8, edgecolor='#e0e0e0', linewidth=0.3, legend=False)

ax1.set_ylabel("Total Variation\nComposition", fontsize=7, fontweight='bold')
ax1.yaxis.set_major_formatter(ticker.PercentFormatter(1.0))
    
# --- BOTTOM: Artifact share of explained variation (strip + box) ---
sns.stripplot(data=df, x="model_label", y="asv", 
              color="#475569", jitter=0.2, size=2.5, alpha=0.3, zorder=2, ax=ax2)

sns.boxplot(data=df, x="model_label", y="asv", color="white", width=0.4, 
            boxprops={'facecolor':'none', 'edgecolor':'#1e293b', 'linewidth':0.8},
            medianprops={'color':'#b91c1c', 'linewidth':1.5}, showfliers=False, zorder=3, ax=ax2)

# --- THE DIVIDERS ---
unique_labels = df["model_label"].unique()
for i in range(1, len(unique_labels)):
    fam_curr = df[df["model_label"] == unique_labels[i]]["family"].iloc[0]
    fam_prev = df[df["model_label"] == unique_labels[i-1]]["family"].iloc[0]
    if fam_curr != fam_prev:
        for ax in [ax1, ax2]:
            ax.axvline(x=i-0.5, color='black', linestyle='-', linewidth=0.6, alpha=0.4)

# 50% ASV threshold and zones
ax2.axhline(0.5, color='black', linestyle='--', linewidth=1, alpha=0.8, zorder=1)
ax2.axhspan(0.5, 1.05, color='#fee2e2', zorder=0) 
ax2.axhspan(-0.05, 0.5, color='#f0fdf4', zorder=0)
ax2.set_ylabel("Artifact Share\nof Explained Variation", fontsize=7, fontweight='bold')
ax2.yaxis.set_major_formatter(ticker.PercentFormatter(1.0))
ax2.set_xlabel("")

# --- FINAL POLISH ---
plt.xticks(rotation=45, ha='right')
# Legend for the components + unexplained variation
refined_labels = ["Baseline\nPersona", "Persona\nWording", "Task\nInstruction", "Item\nWording", "Option\nSymbol", "Unexpl."]
ax1.legend(handles=ax1.get_legend_handles_labels()[0], labels=refined_labels, 
           loc='upper center', bbox_to_anchor=(0.45, 1.25), ncol=6, frameon=False, fontsize=7)

plt.subplots_adjust(hspace=0.08)
plt.savefig(f"../../outputs/figures/component_variance_and_artifact_share_{QUESTIONNAIRE.lower()}.pdf", bbox_inches='tight')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

plt.rcParams.update({'font.family': 'serif', 'font.size': 8})

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 5.5), sharex=True, 
                               gridspec_kw={'height_ratios': [2, 3]})

components = ["baseline_persona", "persona_wording", "task_instruction", "item_wording", "option_symbol"]
df_plot = df.groupby("model_label", sort=False)[components].mean()
df_plot["Unexplained"] = 1.0 - df_plot[components].sum(axis=1)

colors = ["#757575", "#26A69A", "#EF5350", "#5C6BC0", "#FFA726", "#F5F5F5"]

# --- TOP: Total variation composition ---
# Added subtle y-grid underneath the bars (zorder=0)
ax1.grid(axis='y', linestyle=':', alpha=0.6, zorder=0)
df_plot.plot(kind='bar', stacked=True, color=colors, ax=ax1, 
             width=0.8, edgecolor='#e0e0e0', linewidth=0.3, legend=False, zorder=3)

ax1.set_ylabel("Total Variation\nComposition", fontsize=7, fontweight='bold')
ax1.yaxis.set_major_formatter(ticker.PercentFormatter(1.0))
    
# --- BOTTOM: Artifact share of explained variation ---
sns.stripplot(data=df, x="model_label", y="asv", 
              color="#475569", jitter=0.2, size=2.5, alpha=0.3, zorder=2, ax=ax2)

# Increased linewidth from 0.8 to 1.2 for better contrast against background
sns.boxplot(data=df, x="model_label", y="asv", color="white", width=0.4, 
            boxprops={'facecolor':'none', 'edgecolor':'#1e293b', 'linewidth':1.2},
            medianprops={'color':'#b91c1c', 'linewidth':1.5}, showfliers=False, zorder=3, ax=ax2)


unique_labels = df["model_label"].unique()
for i in range(1, len(unique_labels)):
    fam_curr = df[df["model_label"] == unique_labels[i]]["family"].iloc[0]
    fam_prev = df[df["model_label"] == unique_labels[i-1]]["family"].iloc[0]
    if fam_curr != fam_prev:
        for ax in [ax1, ax2]:
            ax.axvline(x=i-0.5, color='black', linestyle='-', linewidth=0.6, alpha=0.4)

# Added alpha=0.3 to prevent background spans from swallowing the data
ax2.axhline(0.5, color='black', linestyle='--', linewidth=1, alpha=0.8, zorder=1)
ax2.axhspan(0.5, 1.05, color='#fee2e2', alpha=0.3, zorder=0) 
ax2.axhspan(-0.05, 0.5, color='#f0fdf4', alpha=0.3, zorder=0)
ax2.set_ylabel("Artifact Share\nof Explained Variation", fontsize=9, fontweight='bold')
ax2.yaxis.set_major_formatter(ticker.PercentFormatter(1.0))
ax2.set_xlabel("")


plt.xticks(rotation=45, ha='right')

# Stacked legend in 3 columns across 2 rows to prevent horizontal collision
refined_labels = ["Baseline\nPersona", "Persona\nWording", "Task\nInstruction", "Item\nWording", "Option\nSymbol", "Unexpl."]
ax1.legend(handles=ax1.get_legend_handles_labels()[0], labels=refined_labels, 
           loc='lower center', bbox_to_anchor=(0.5, 1.05), ncol=6, frameon=False, fontsize=7)

plt.subplots_adjust(hspace=0.08)
plt.savefig(f"../../outputs/figures/component_variance_and_artifact_share_{QUESTIONNAIRE.lower()}.pdf", bbox_inches='tight')